# 01 v4 (run_02) - split by PATIENT and by TIME (as discussed with Prof. Dutta); PLOS threshold from training stays only

Following the example in the meeting minutes (page 53): patients are split at random, and each split may only use
stays that fall in its own time window.

1. **Patients:** shuffled with seed 42 and divided into train / validation / test patients (`PATIENT_FRACS`).
2. **Time:** one cut-off date `T_CUT` (a quantile of all admission dates, `CUT_Q`).
   - Training examples = stays of **training patients** admitted **before** `T_CUT`.
   - Validation and test examples = stays of **validation / test patients** admitted **on or after** `T_CUT`.
   - Example from the minutes: P1, P2 train and P3 test; P3's stay from 2010 is before the cut-off, so it is not a test example.
3. **Excluded stays:** training patients' stays after `T_CUT` (future data) are dropped. Validation and test patients' stays
   before `T_CUT` are not targets; they are saved separately in `history.parquet`, so sequence models can use them as
   earlier visits (input only), never as prediction targets.
4. **PLOS threshold:** mean + 2 x SD of length of stay over training stays only.
5. Cell 3a prints split sizes for several cut-offs and patient fractions, so the choice can be made before saving.

Optional: `EXCLUDE_COVID = True` removes COVID-19 stays before everything else (to run the study with and without them).
Everything saved keeps the same contract (`train/val/test.parquet`, `meta.json`), so later notebooks run unchanged.
The previous outputs are copied to `genai_random_patient_split/` before anything is overwritten.

In [ ]:
# Cell 1 - paths and settings
from google.colab import drive
drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/genai_synthea/run_02"
DATA = f"{BASE}/processed/Dataset_Tabular.csv"
RAW  = f"{BASE}/raw/csv"
OUT  = f"{BASE}/genai"
SEED = 42
TOP_REASONS = 20
PATIENT_FRACS = (0.60, 0.20, 0.20)   # train / validation / test patients (random, by patient)
CUT_Q = 0.75                         # time cut-off = this quantile of all admission dates
EXCLUDE_COVID = False                # True = study without COVID-19 stays
import os, shutil
if os.path.exists(f"{OUT}/meta.json") and not os.path.exists(f"{BASE}/genai_random_patient_split"):
    shutil.copytree(OUT, f"{BASE}/genai_random_patient_split"); print("Backed up the previous split to genai_random_patient_split/")

In [ ]:
# Cell 2 — load, attach encounter type/reason
import os, json, numpy as np, pandas as pd
os.makedirs(OUT, exist_ok=True)
df = pd.read_csv(DATA, low_memory=False)
enc = pd.read_csv(f"{RAW}/encounters.csv", usecols=["Id", "ENCOUNTERCLASS", "START", "DESCRIPTION", "REASONDESCRIPTION"])
enc = enc[enc["ENCOUNTERCLASS"] == "inpatient"].rename(columns={
    "Id": "hadm_id", "DESCRIPTION": "encounter_type", "REASONDESCRIPTION": "encounter_reason"})
enc["admittime"] = pd.to_datetime(enc.START, errors="coerce", utc=True).dt.tz_localize(None)
df = df.merge(enc[["hadm_id", "admittime", "encounter_type", "encounter_reason"]], on="hadm_id", how="left")
df["target"] = pd.to_numeric(df["target"], errors="coerce")
df = df.dropna(subset=["target"]).copy()
df = df.dropna(subset=["admittime"]).copy()
if EXCLUDE_COVID:
    cov = df.encounter_reason.fillna("").str.lower().str.contains("coronavirus|covid")
    print(f"Excluding {cov.sum():,} COVID-19 stays"); df = df[~cov].copy()
print(f"Rows {len(df):,} | patients {df['subject_id'].nunique():,} | admissions from {df.admittime.min():%Y-%m-%d} to {df.admittime.max():%Y-%m-%d}")
df_all = df.copy()   # untouched copy: Cells 3a and 3 always start from it, so re-running them is safe


In [ ]:
# Cell 3a - split sizes for several cut-offs and patient fractions (choose CUT_Q / PATIENT_FRACS in Cell 1 from this)
df = df_all.copy()
rng = np.random.default_rng(SEED)
pats = df.subject_id.unique().copy(); rng.shuffle(pats)
def assign(fracs):
    n = len(pats); a, b = int(fracs[0] * n), int((fracs[0] + fracs[1]) * n)
    return {**{p: "train" for p in pats[:a]}, **{p: "val" for p in pats[a:b]}, **{p: "test" for p in pats[b:]}}
_mean, _sd = df.target.mean(), df.target.std()
rows = []
for fr in [(0.70, 0.15, 0.15), (0.60, 0.20, 0.20), (0.50, 0.25, 0.25)]:
    grp = df.subject_id.map(assign(fr))
    for q in (0.65, 0.70, 0.75, 0.80):
        t = df.admittime.quantile(q); before = df.admittime < t
        tr = df[(grp == "train") & before]; thr = tr.target.mean() + 2 * tr.target.std()
        va = df[(grp == "val") & ~before]; te = df[(grp == "test") & ~before]
        rows.append({"patients train/val/test": "/".join(f"{x:.2f}" for x in fr), "CUT_Q": q, "cut-off date": f"{t:%Y-%m-%d}",
                     "train stays": len(tr), "train PLOS": int((tr.target > thr).sum()), "val stays": len(va), "val PLOS": int((va.target > thr).sum()),
                     "test stays": len(te), "test PLOS": int((te.target > thr).sum()), "dropped or history-only": len(df) - len(tr) - len(va) - len(te)})
print(f"All stays {len(df):,}, admitted {df.admittime.min():%Y-%m-%d} to {df.admittime.max():%Y-%m-%d}")
pd.DataFrame(rows)

In [ ]:
# Cell 3 - split by patient AND time with the chosen settings, then the PLOS threshold from training stays only
df = df_all.copy()
grp = df.subject_id.map(assign(PATIENT_FRACS))
T_CUT = df.admittime.quantile(CUT_Q); before = df.admittime < T_CUT
df["split"] = np.select([(grp == "train") & before, (grp == "val") & ~before, (grp == "test") & ~before],
                        ["train", "val", "test"], default="excluded")
df["role"] = np.select([df.split != "excluded", (grp != "train") & before], ["target", "history"], default="future_dropped")
assert len(df) == len(df_all)
print(f"Time cut-off {T_CUT:%Y-%m-%d} (quantile {CUT_Q}) | patients {PATIENT_FRACS}")
print(df.groupby([grp.rename('patient group'), 'role']).size().unstack(fill_value=0))
assert df[df.split != "excluded"].groupby("subject_id").split.nunique().max() == 1          # no patient in two splits
assert df[df.split == "train"].admittime.max() < T_CUT <= min(df[df.split == "val"].admittime.min(), df[df.split == "test"].admittime.min())
history = df[df.role == "history"].copy()                                                    # earlier stays of val/test patients (input only)
df = df[df.split != "excluded"].copy()
tr_los = df.loc[df.split == "train", "target"]
PLOS_THRESHOLD = float(tr_los.mean() + 2 * tr_los.std())
THR_ALL = float(df.target.mean() + 2 * df.target.std())
df["is_plos"] = (df["target"] > PLOS_THRESHOLD).astype(int)
history["is_plos"] = (history["target"] > PLOS_THRESHOLD).astype(int)
print(f"PLOS threshold (mean + 2 SD, training stays): {PLOS_THRESHOLD:.2f} d  [all kept stays would give {THR_ALL:.2f} d]")
for s in ("train", "val", "test"):
    d = df[df.split == s]
    print(f"{s:5s}: {d.subject_id.nunique():6,} patients | {len(d):6,} stays | {d.admittime.min():%Y-%m-%d} to {d.admittime.max():%Y-%m-%d} | "
          f"PLOS {d.is_plos.sum():4d} ({d.is_plos.mean():.1%})")
print(f"history-only stays of val/test patients: {len(history):,}")
cov = df.encounter_reason.fillna("").str.lower().str.contains("covid|coronavirus")
print("COVID-19 stays per split (plos 0/1):"); print(pd.crosstab(df.split, [cov.rename("covid"), df.is_plos.rename("plos")]))


In [ ]:
# Cell 3b - how other common PLOS definitions would compare (training stays only; for the report, nothing below changes)
y = df.loc[df.split == "train", "target"]; ly = np.log(y.clip(lower=1e-3))
q1, q3 = y.quantile([0.25, 0.75])
alts = {"mean + 2 SD (lab rule, USED)": y.mean() + 2 * y.std(),
        "geometric mean + 2 SD (log scale; DRG trim point)": float(np.exp(ly.mean() + 2 * ly.std())),
        "Q3 + 1.5 x IQR (Tukey outlier fence)": q3 + 1.5 * (q3 - q1),
        "90th percentile": y.quantile(0.90), "95th percentile": y.quantile(0.95),
        "fixed 7 days": 7.0, "fixed 14 days": 14.0}
alt = pd.DataFrame({"threshold (days)": alts}); alt["share of training stays above"] = [(y > v).mean() for v in alt["threshold (days)"]]
alt["share of ALL stays above"] = [(df.target > v).mean() for v in alt["threshold (days)"]]
print(f"Training stays {len(y):,} | mean {y.mean():.2f} d | SD {y.std():.2f} d | median {y.median():.2f} d | skewness {y.skew():.2f}")
alt.round(3)

In [ ]:
# Cell 4 - reason-based modes as binary columns
REASON_MODES = {
    "addiction":       ["Dependent drug abuse"],
    "lung_cancer":     ["Non-small cell carcinoma of lung", "small cell malignant neoplasm of lung"],
    "cardiopulmonary": ["congestive heart failure", "obstructive bronchitis", "emphysema"],
    "onc_transplant":  ["transplantation", "leukemia", "neoplasm of colon"],
    "infectious":      ["coronavirus"],
}
r = df["encounter_reason"].fillna("").str.lower()
MODE_COLS = []
for k, kws in REASON_MODES.items():
    col = f"mode_{k}"
    df[col] = r.apply(lambda s: any(w.lower() in s for w in kws)).astype("int8")
    MODE_COLS.append(col)
unexpl = (df[MODE_COLS].sum(axis=1) == 0)
print(pd.DataFrame({"plos_n": df.loc[df.is_plos == 1, MODE_COLS].sum(),
                    "plos_share": df.loc[df.is_plos == 1, MODE_COLS].mean()}).round(3))
print(f"Unexplained PLOS: {unexpl[df.is_plos == 1].mean():.1%}")

In [ ]:
# Cell 5 - features
TEXT_COLS = ["hpi", "social_history", "chief_complaint", "vitals", "labs", "conditions", "medications"]
df = df.drop(columns=[c for c in TEXT_COLS + ["birthdate"] if c in df.columns])
ID_COLS  = ["subject_id", "hadm_id"]
NUM_COLS = ["anchor_age"]
top = df["encounter_reason"].value_counts().head(TOP_REASONS).index
df["encounter_reason"] = df["encounter_reason"].where(df["encounter_reason"].isin(top), "other").fillna("other")
df["encounter_type"] = df["encounter_type"].fillna("unknown")
CAT_COLS = [c for c in ["gender", "race", "ethnicity", "admission_type", "encounter_type", "encounter_reason"] if c in df.columns]
for c in CAT_COLS: df[c] = df[c].astype(str)
COND_COLS = [c for c in df.columns if c.startswith("snomed_")]
df[COND_COLS] = df[COND_COLS].fillna(0).astype("int8")
BIN_COLS = COND_COLS + MODE_COLS
df["anchor_age"] = pd.to_numeric(df["anchor_age"], errors="coerce")
print(f"{len(NUM_COLS)} numeric | {len(CAT_COLS)} categorical {CAT_COLS} | {len(COND_COLS)} condition flags + {len(MODE_COLS)} mode flags")

In [ ]:
# Cell 6 - the three splits
train, val, test = (df[df.split == s].copy() for s in ["train", "val", "test"])
for name, d in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s}: {d['subject_id'].nunique():5d} patients | {len(d):6d} stays | PLOS {d['is_plos'].sum():4d} ({d['is_plos'].mean():.1%}) | "
          f"rare modes in PLOS: infectious {int(d.loc[d.is_plos==1,'mode_infectious'].sum())}, onc_transplant {int(d.loc[d.is_plos==1,'mode_onc_transplant'].sum())}")

In [ ]:
# Cell 7 - drop columns with no information in TRAIN
const_cat = [c for c in CAT_COLS if train[c].nunique() <= 1]
zero_bin  = [c for c in COND_COLS if train[c].sum() == 0]
for d in (train, val, test): d.drop(columns=const_cat + zero_bin, inplace=True)
CAT_COLS  = [c for c in CAT_COLS if c not in const_cat]
COND_COLS = [c for c in COND_COLS if c not in zero_bin]
BIN_COLS  = COND_COLS + MODE_COLS
print(f"Dropped constant: {const_cat} | all-zero condition flags: {len(zero_bin)}")

In [ ]:
# Cell 8 - save splits + meta (same contract, so later notebooks run unchanged)
cols = ID_COLS + NUM_COLS + CAT_COLS + BIN_COLS + ["target", "is_plos"]
for name, d in [("train", train), ("val", val), ("test", test)]:
    d[cols].to_parquet(f"{OUT}/{name}.parquet", index=False)
meta = {"source": DATA, "run": "run_02", "plos_threshold": PLOS_THRESHOLD, "seed": SEED,
        "split": "patient + time: random patients (PATIENT_FRACS), train = train patients before T_CUT, val/test = val/test patients on or after T_CUT",
        "patient_fracs": PATIENT_FRACS, "cut_q": CUT_Q, "time_cutoff": str(T_CUT), "exclude_covid": EXCLUDE_COVID, "history_stays": int(len(history)), "threshold_rule": "mean + 2 SD of LOS, training stays", "threshold_all_stays": THR_ALL,
        "id_cols": ID_COLS, "num_cols": NUM_COLS, "cat_cols": CAT_COLS, "bin_cols": BIN_COLS,
        "cond_cols": COND_COLS, "target": "target", "label": "is_plos",
        "modes": {c.replace("mode_", ""): [c] for c in MODE_COLS}, "reason_modes": REASON_MODES,
        "sizes": {k: {"stays": int(len(d)), "patients": int(d["subject_id"].nunique()), "plos": int(d["is_plos"].sum())}
                  for k, d in [("train", train), ("val", val), ("test", test)]}}
history.drop(columns=[c for c in history.columns if c not in cols + ["admittime"]], errors="ignore").to_parquet(f"{OUT}/history.parquet", index=False)
json.dump(meta, open(f"{OUT}/meta.json", "w"), indent=2)
print("Saved to", OUT); print(json.dumps(meta["sizes"], indent=2))